In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import vectorbt as vbt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys

sys.path.insert(0, str(Path('.').resolve().parent))
from dema_strat import load_ohlcv, vbt_frame, free
from constants import CASH, COMMISSION, SESSIONS

TABLE_NAME = "f1_session"
SMOOTH_PERIOD = 12
RISK_PER_TRADE = 100.0


In [ ]:
def load_data(file_path, *, verbose=False, use_vwap=False):
    return load_ohlcv(file_path, use_cache=True, add_vwap=use_vwap, verbose=verbose)

def session_mask(index, session):
    start = pd.Timestamp(session["start"]).time()
    end = pd.Timestamp(session["end"]).time()
    tod = np.array(index.to_series().dt.time.values)
    if start <= end:
        return (tod >= start) & (tod < end)
    return (tod >= start) | (tod < end)

def make_signals(data, n1=9, atr_period=14, sl_mult=1.0, rr_ratio=0.5, use_vwap=False, smooth_period=SMOOTH_PERIOD):
    close = data["Close"]
    high = data["High"]
    low = data["Low"]

    ema1 = vbt.indicators.MA.run(close, window=n1, ewm=True)
    smoothed = vbt.indicators.MA.run(ema1.ma, window=smooth_period, ewm=True)
    atr = vbt.indicators.ATR.run(high, low, close, window=atr_period)

    sl_dist = sl_mult * atr.atr
    tp_dist = rr_ratio * sl_dist
    sl_pct = sl_dist / close
    tp_pct = tp_dist / close

    long_mask = (smoothed.ma > ema1.ma) & np.isfinite(atr.atr) & (atr.atr > 0)
    short_mask = (ema1.ma > smoothed.ma) & np.isfinite(atr.atr) & (atr.atr > 0)

    if use_vwap:
        long_mask = long_mask & (close > data["VWAP"])
        short_mask = short_mask & (close < data["VWAP"])

    return pd.DataFrame({
        "ema1": ema1.ma,
        "smoothed": smoothed.ma,
        "atr": atr.atr,
        "sl_pct": sl_pct,
        "tp_pct": tp_pct,
        "long": long_mask,
        "short": short_mask,
    })

def run_backtest(data, n1=9, atr_period=14, sl_mult=1.0, rr_ratio=0.5, use_vwap=False, smooth_period=SMOOTH_PERIOD, session=None):
    signals = make_signals(data, n1=n1, atr_period=atr_period, sl_mult=sl_mult, rr_ratio=rr_ratio, use_vwap=use_vwap, smooth_period=smooth_period)
    freq = data.index.to_series().diff().median()

    long_signal = signals["long"].to_numpy(dtype=bool)
    short_signal = signals["short"].to_numpy(dtype=bool)

    n = len(data)
    in_session = np.ones(n, dtype=bool)
    session_end = np.zeros(n, dtype=bool)
    if session is not None:
        in_session = session_mask(data.index, session)
        next_in = np.empty(n, dtype=bool)
        next_in[:-1] = in_session[1:]
        next_in[-1] = False
        session_end = in_session & ~next_in

    long_signal = long_signal & in_session & ~session_end
    short_signal = short_signal & in_session & ~session_end
    exits_long = short_signal | session_end
    exits_short = long_signal | session_end

    close_np = data["Close"].to_numpy(dtype=np.float64)

    sl_pct_np = signals["sl_pct"].to_numpy(dtype=np.float64)
    size_value = np.where(sl_pct_np > 0, RISK_PER_TRADE / sl_pct_np, 0.0)

    portfolio = vbt.Portfolio.from_signals(
        close=close_np,
        entries=long_signal,
        exits=exits_long,
        short_entries=short_signal,
        short_exits=exits_short,
        init_cash=CASH,
        fees=COMMISSION,
        size=size_value,
        size_type="value",
        freq=freq,
        sl_stop=signals["sl_pct"].to_numpy(dtype=np.float64),
        tp_stop=signals["tp_pct"].to_numpy(dtype=np.float64),
    )

    return portfolio, portfolio.stats(), signals

In [ ]:
file_path = Path("~/market_data/XAUUSD/XAUUSD_M15.csv").expanduser()
data = load_data(file_path, verbose=True, use_vwap=True)

portfolio, stats, signals = run_backtest(
    data,
    n1=5,
    rr_ratio=2,
    atr_period=18,
    session=SESSIONS[1],
    sl_mult=3,
    use_vwap=True,
    smooth_period=12,
)
print(stats)

trades = portfolio.trades.records_readable
orders = portfolio.orders.records_readable
print(f"Trades: {len(trades)}")
print(f"Orders: {len(orders)}")

In [ ]:
# --- OHLCV Candlestick with EMA signals ---
def plot_ohlcv_signals(data, signals, portfolio, title="OHLCV with Signals", height=800):
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        row_heights=[0.6, 0.2, 0.2],
        subplot_titles=("Price & Signals", "EMA & Smoothed EMA", "ATR")
    )

    # Candlesticks
    fig.add_trace(
        go.Candlestick(
            x=data.index,
            open=data["Open"],
            high=data["High"],
            low=data["Low"],
            close=data["Close"],
            name="OHLCV",
            increasing_line_color="#26a69a",
            decreasing_line_color="#ef5350",
        ),
        row=1, col=1
    )

    # VWAP if available
    if "VWAP" in data.columns:
        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=data["VWAP"],
                mode="lines",
                name="VWAP",
                line=dict(color="#ff9800", width=1),
                opacity=0.7
            ),
            row=1, col=1
        )

    # Long entries
    long_entries = signals[signals["long"]].index
    if len(long_entries) > 0:
        fig.add_trace(
            go.Scatter(
                x=long_entries,
                y=data.loc[long_entries, "Low"] * 0.999,
                mode="markers",
                name="Long Entry",
                marker=dict(symbol="triangle-up", size=10, color="#26a69a", line=dict(width=1, color="white")),
            ),
            row=1, col=1
        )

    # Short entries
    short_entries = signals[signals["short"]].index
    if len(short_entries) > 0:
        fig.add_trace(
            go.Scatter(
                x=short_entries,
                y=data.loc[short_entries, "High"] * 1.001,
                mode="markers",
                name="Short Entry",
                marker=dict(symbol="triangle-down", size=10, color="#ef5350", line=dict(width=1, color="white")),
            ),
            row=1, col=1
        )

    # EMA1
    fig.add_trace(
        go.Scatter(
            x=signals.index,
            y=signals["ema1"],
            mode="lines",
            name="EMA1",
            line=dict(color="#2196f3", width=1),
        ),
        row=2, col=1
    )

    # Smoothed EMA
    fig.add_trace(
        go.Scatter(
            x=signals.index,
            y=signals["smoothed"],
            mode="lines",
            name="Smoothed EMA",
            line=dict(color="#9c27b0", width=1),
        ),
        row=2, col=1
    )

    # ATR
    fig.add_trace(
        go.Scatter(
            x=signals.index,
            y=signals["atr"],
            mode="lines",
            name="ATR",
            line=dict(color="#ff5722", width=1),
            fill="tozeroy",
            fillcolor="rgba(255,87,34,0.1)"
        ),
        row=3, col=1
    )

    fig.update_layout(
        title=title,
        height=height,
        xaxis_rangeslider_visible=False,
        template="plotly_dark",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=50, r=50, t=80, b=50)
    )
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="EMA", row=2, col=1)
    fig.update_yaxes(title_text="ATR", row=3, col=1)
    
    return fig

fig = plot_ohlcv_signals(data, signals, portfolio, title="XAUUSD M15 - DEMA Strategy Signals (LONDON Session)")
fig.show()

In [ ]:
# --- Portfolio Value ---
def plot_portfolio_value(portfolio, title="Portfolio Value", height=500):
    value = portfolio.value()
    
    fig = go.Figure()
    
    fig.add_trace(
        go.Scatter(
            x=value.index,
            y=value.values,
            mode="lines",
            name="Portfolio Value",
            line=dict(color="#4caf50", width=2),
            fill="tozeroy",
            fillcolor="rgba(76,175,80,0.1)"
        )
    )

    # Add initial cash line
    fig.add_hline(y=CASH, line_dash="dash", line_color="#9e9e9e", annotation_text=f"Initial Cash: ${CASH:,.0f}")

    fig.update_layout(
        title=title,
        height=height,
        template="plotly_dark",
        xaxis_title="Time",
        yaxis_title="Value ($)",
        hovermode="x unified",
        margin=dict(l=50, r=50, t=80, b=50)
    )
    
    return fig

fig = plot_portfolio_value(portfolio, title="Portfolio Equity Curve")
fig.show()

In [ ]:
# --- Drawdown ---
def plot_drawdown(portfolio, title="Drawdown", height=400):
    dd = portfolio.drawdown()
    
    fig = go.Figure()
    
    fig.add_trace(
        go.Scatter(
            x=dd.index,
            y=dd.values * 100,
            mode="lines",
            name="Drawdown %",
            line=dict(color="#f44336", width=1),
            fill="tozeroy",
            fillcolor="rgba(244,67,54,0.2)"
        )
    )

    fig.update_layout(
        title=title,
        height=height,
        template="plotly_dark",
        xaxis_title="Time",
        yaxis_title="Drawdown (%)",
        hovermode="x unified",
        margin=dict(l=50, r=50, t=80, b=50)
    )
    
    return fig

fig = plot_drawdown(portfolio, title="Portfolio Drawdown")
fig.show()

In [ ]:
# --- Individual Trades ---
def plot_trades(data, portfolio, max_trades=50, title="Trades", height=600):
    trades = portfolio.trades.records_readable
    
    if len(trades) == 0:
        print("No trades to display")
        return None
    
    # Limit to max_trades for visibility
    trades_plot = trades.head(max_trades) if len(trades) > max_trades else trades
    
    fig = make_subplots(
        rows=len(trades_plot), cols=1,
        shared_xaxes=True,
        vertical_spacing=0.02,
        subplot_titles=[f"Trade {i}: {row['Direction']} | PnL: ${row['PnL']:.2f} | {row['Return']*100:.2f}%" for i, row in trades_plot.iterrows()]
    )

    for idx, (_, trade) in enumerate(trades_plot.iterrows()):
        row = idx + 1
        entry_time = pd.Timestamp(trade["Entry Timestamp"])
        exit_time = pd.Timestamp(trade["Exit Timestamp"])
        
        # Get price data for trade duration
        trade_data = data.loc[entry_time:exit_time]
        
        # Candlesticks for this trade
        fig.add_trace(
            go.Candlestick(
                x=trade_data.index,
                open=trade_data["Open"],
                high=trade_data["High"],
                low=trade_data["Low"],
                close=trade_data["Close"],
                name=f"Trade {idx}",
                showlegend=False,
                increasing_line_color="#26a69a",
                decreasing_line_color="#ef5350",
            ),
            row=row, col=1
        )

        # Entry marker
        fig.add_trace(
            go.Scatter(
                x=[entry_time],
                y=[trade["Avg Entry Price"]],
                mode="markers",
                marker=dict(symbol="triangle-up" if trade["Direction"] == "Long" else "triangle-down", size=12, color="#fff", line=dict(width=2, color="#26a69a" if trade["Direction"] == "Long" else "#ef5350")),
                name="Entry",
                showlegend=False,
            ),
            row=row, col=1
        )

        # Exit marker
        fig.add_trace(
            go.Scatter(
                x=[exit_time],
                y=[trade["Avg Exit Price"]],
                mode="markers",
                marker=dict(symbol="x", size=12, color="#fff", line=dict(width=2, color="#ef5350" if trade["PnL"] > 0 else "#26a69a")),
                name="Exit",
                showlegend=False,
            ),
            row=row, col=1
        )

    fig.update_layout(
        title=f"{title} (showing first {min(max_trades, len(trades))} of {len(trades)} trades)",
        height=height,
        template="plotly_dark",
        xaxis_rangeslider_visible=False,
        margin=dict(l=50, r=50, t=80, b=50)
    )

    return fig

fig = plot_trades(data, portfolio, max_trades=20)
if fig:
    fig.show()

In [ ]:
# --- Trade Distribution ---
def plot_trade_distribution(portfolio, title="Trade Distribution", height=500):
    trades = portfolio.trades.records_readable
    
    if len(trades) == 0:
        print("No trades to display")
        return None
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=("PnL Distribution", "Returns Distribution", "Trade Duration", "Cumulative PnL"),
        vertical_spacing=0.1,
        horizontal_spacing=0.1
    )

    # PnL histogram
    fig.add_trace(
        go.Histogram(
            x=trades["PnL"],
            nbinsx=50,
            name="PnL",
            marker_color="#2196f3",
            opacity=0.7
        ),
        row=1, col=1
    )

    # Returns histogram
    fig.add_trace(
        go.Histogram(
            x=trades["Return"] * 100,
            nbinsx=50,
            name="Return %",
            marker_color="#9c27b0",
            opacity=0.7
        ),
        row=1, col=2
    )

    # Duration histogram (in bars)
    durations = trades["Exit Timestamp"] - trades["Entry Timestamp"]
    fig.add_trace(
        go.Histogram(
            x=durations,
            nbinsx=50,
            name="Duration (bars)",
            marker_color="#ff9800",
            opacity=0.7
        ),
        row=2, col=1
    )

    # Cumulative PnL
    cum_pnl = trades["PnL"].cumsum()
    fig.add_trace(
        go.Scatter(
            x=list(range(len(cum_pnl))),
            y=cum_pnl.values,
            mode="lines",
            name="Cumulative PnL",
            line=dict(color="#4caf50", width=2),
        ),
        row=2, col=2
    )

    fig.update_layout(
        title=title,
        height=height,
        template="plotly_dark",
        showlegend=False,
        margin=dict(l=50, r=50, t=80, b=50)
    )

    fig.update_xaxes(title_text="PnL ($)", row=1, col=1)
    fig.update_xaxes(title_text="Return (%)", row=1, col=2)
    fig.update_xaxes(title_text="Duration (bars)", row=2, col=1)
    fig.update_xaxes(title_text="Trade #", row=2, col=2)
    fig.update_yaxes(title_text="Count", row=1, col=1)
    fig.update_yaxes(title_text="Count", row=1, col=2)
    fig.update_yaxes(title_text="Count", row=2, col=1)
    fig.update_yaxes(title_text="Cumulative PnL ($)", row=2, col=2)

    return fig

fig = plot_trade_distribution(portfolio, title="Trade Analysis")
fig.show()

In [ ]:
# --- Orders Visualization ---
def plot_orders(data, portfolio, title="Orders", height=600):
    orders = portfolio.orders.records_readable
    
    if len(orders) == 0:
        print("No orders to display")
        return None
    
    fig = go.Figure()
    
    # Price line
    fig.add_trace(
        go.Scatter(
            x=data.index,
            y=data["Close"],
            mode="lines",
            name="Close Price",
            line=dict(color="#9e9e9e", width=1),
            opacity=0.5
    )
    )

    # Buy orders
    buy_orders = orders[orders["Side"] == "Buy"]
    if len(buy_orders) > 0:
        fig.add_trace(
            go.Scatter(
                x=buy_orders["Timestamp"],
                y=buy_orders["Price"],
                mode="markers",
                name="Buy Order",
                marker=dict(symbol="triangle-up", size=10, color="#26a69a", line=dict(width=1, color="white")),
                text=[f"Size: {s:.4f}<br>Price: {p:.2f}" for s, p in zip(buy_orders["Size"], buy_orders["Price"])],
                hovertemplate="%{text}<extra></extra>",
            )
        )

    # Sell orders
    sell_orders = orders[orders["Side"] == "Sell"]
    if len(sell_orders) > 0:
        fig.add_trace(
            go.Scatter(
                x=sell_orders["Timestamp"],
                y=sell_orders["Price"],
                mode="markers",
                name="Sell Order",
                marker=dict(symbol="triangle-down", size=10, color="#ef5350", line=dict(width=1, color="white")),
                text=[f"Size: {s:.4f}<br>Price: {p:.2f}" for s, p in zip(sell_orders["Size"], sell_orders["Price"])],
                hovertemplate="%{text}<extra></extra>",
            )
        )

    fig.update_layout(
        title=title,
        height=height,
        template="plotly_dark",
        xaxis_title="Time",
        yaxis_title="Price",
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=50, r=50, t=80, b=50)
    )

    return fig

fig = plot_orders(data, portfolio, title="Order Execution")
fig.show()

In [ ]:
# --- Monthly Returns Heatmap ---
def plot_monthly_returns(portfolio, title="Monthly Returns", height=500):
    returns = portfolio.returns()
    
    # Ensure DatetimeIndex for resampling
    if not isinstance(returns.index, pd.DatetimeIndex):
        returns.index = pd.to_datetime(returns.index)
    
    # Resample to monthly
    monthly = returns.resample("ME").apply(lambda x: (1 + x).prod() - 1)
    monthly_pct = monthly * 100
    
    # Create pivot table for heatmap
    monthly_df = pd.DataFrame({
        "year": monthly_pct.index.year,
        "month": monthly_pct.index.month,
        "return": monthly_pct.values
    })
    
    pivot = monthly_df.pivot(index="year", columns="month", values="return")
    pivot = pivot.reindex(columns=range(1, 13))
    pivot.columns = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    fig = go.Figure(data=go.Heatmap(
        z=pivot.values,
        x=pivot.columns,
        y=pivot.index.astype(str),
        colorscale="RdYlGn",
        zmid=0,
        text=np.round(pivot.values, 2),
        texttemplate="%{text}%",
        textfont={"size": 10},
        colorbar=dict(title="Return %")
    ))

    fig.update_layout(
        title=title,
        height=height,
        template="plotly_dark",
        margin=dict(l=50, r=50, t=80, b=50)
    )

    return fig

fig = plot_monthly_returns(portfolio, title="Monthly Returns Heatmap")
fig.show()

In [ ]:
# --- Rolling Metrics ---
def plot_rolling_metrics(portfolio, window=100, title="Rolling Metrics", height=600):
    returns = portfolio.returns()
    trades = portfolio.trades.records_readable
    
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=(f"Rolling Sharpe ({window} trades)", f"Rolling Win Rate ({window} trades)", f"Rolling Profit Factor ({window} trades)")
    )

    if len(trades) > window:
        # Rolling Sharpe
        rolling_sharpe = returns.rolling(window).apply(lambda x: np.sqrt(252*96) * x.mean() / x.std() if x.std() > 0 else 0)
        fig.add_trace(
            go.Scatter(x=rolling_sharpe.index, y=rolling_sharpe.values, mode="lines", name="Rolling Sharpe", line=dict(color="#2196f3")),
            row=1, col=1
        )

        # Rolling Win Rate
        win_rate = trades["Return"].rolling(window).apply(lambda x: (x > 0).mean() * 100)
        fig.add_trace(
            go.Scatter(x=trades.index, y=win_rate.values, mode="lines", name="Win Rate %", line=dict(color="#9c27b0")),
            row=2, col=1
        )
        fig.add_hline(y=50, line_dash="dash", line_color="#9e9e9e", row=2, col=1)

        # Rolling Profit Factor
        def profit_factor(x):
            wins = x[x > 0].sum()
            losses = abs(x[x < 0].sum())
            return wins / losses if losses > 0 else np.nan
        
        rolling_pf = trades["Return"].rolling(window).apply(profit_factor)
        fig.add_trace(
            go.Scatter(x=trades.index, y=rolling_pf.values, mode="lines", name="Profit Factor", line=dict(color="#ff9800")),
            row=3, col=1
        )
        fig.add_hline(y=1, line_dash="dash", line_color="#9e9e9e", row=3, col=1)

    fig.update_layout(
        title=title,
        height=height,
        template="plotly_dark",
        showlegend=False,
        margin=dict(l=50, r=50, t=80, b=50)
    )

    fig.update_yaxes(title_text="Sharpe", row=1, col=1)
    fig.update_yaxes(title_text="Win Rate %", row=2, col=1)
    fig.update_yaxes(title_text="Profit Factor", row=3, col=1)

    return fig

fig = plot_rolling_metrics(portfolio, window=100)
fig.show()

In [ ]:
# --- Equity Components Breakdown ---
def plot_equity_components(portfolio, title="Equity Components", height=500):
    value = portfolio.value()
    cash = portfolio.cash()
    assets = portfolio.assets()
    
    fig = go.Figure()
    
    fig.add_trace(
        go.Scatter(
            x=value.index,
            y=value.values,
            mode="lines",
            name="Total Value",
            line=dict(color="#4caf50", width=2),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=cash.index,
            y=cash.values,
            mode="lines",
            name="Cash",
            line=dict(color="#2196f3", width=1),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=assets.index,
            y=assets.values,
            mode="lines",
            name="Assets Value",
            line=dict(color="#ff9800", width=1),
        )
    )

    fig.update_layout(
        title=title,
        height=height,
        template="plotly_dark",
        xaxis_title="Time",
        yaxis_title="Value ($)",
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=50, r=50, t=80, b=50)
    )

    return fig

fig = plot_equity_components(portfolio)
fig.show()

In [ ]:
free(data, signals)